# Silver Transformation - dimpaymenttypes

This notebook builds the `dimpaymenttypes` dataset in the Silver layer.



## Run Shared Libraries



In [ ]:
%run ../Misc/sharedlibraries

In [ ]:
UpdatedDateTime = datetime.datetime.now()
Entity = "dimpaymenttypes"

## Read Source Tables



In [ ]:
salesorderlinedf = spark.table("bronze.salesorderline")
display(salesorderlinedf)

In [ ]:
# payment_counts_df = salesorderlinedf.groupBy("PaymentTypeDesc").count()
# display(payment_counts_df)

## Create Silver Table Structure



In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS silver.dimpaymenttypes(
        PaymentTypeId INT,
        PaymentTypeDesc STRING
)
USING DELTA

## Build Silver Dataset



In [ ]:
df = (
    salesorderlinedf
    .select(F.trim(F.col("PaymentTypeDesc")).alias("PaymentTypeDesc"))
    .filter(F.col("PaymentTypeDesc").isNotNull())
    .distinct()
)

display(df)

In [ ]:
paymenttypedf = spark.table("silver.dimpaymenttypes")
display(paymenttypedf)

In [ ]:
newrowsdf = df.exceptAll(paymenttypedf.select("PaymentTypeDesc"))
display(newrowsdf)

In [ ]:
maxdf = spark.sql("select ifnull(max(PaymentTypeId),0) as maxid from {df}", df=paymenttypedf)

toprow = maxdf.head(1)
maxid = toprow[0][0]

print(maxid)

In [ ]:
import pyspark.sql.window as W

In [ ]:
idsdf = newrowsdf.withColumn(
    "PaymentTypeId",
    F.row_number().over(window=W.Window.orderBy(F.col("PaymentTypeDesc")))
)

display(idsdf)

In [ ]:
idsFinal = idsdf.withColumn(
    "PaymentTypeId",
    (F.col("PaymentTypeId") + F.lit(maxid)).cast("int")
)

display(idsFinal)

In [ ]:
# %sql
# select * from silver.dimpaymenttypes

## Prepare Final DataFrame



In [ ]:
df_final = idsFinal.select(
    F.col("PaymentTypeId").cast("int").alias("PaymentTypeId"),
    F.col("PaymentTypeDesc").cast("string").alias("PaymentTypeDesc")
)

display(df_final)

## Verify Source Schema



In [ ]:
appendToDeltaTable(df_final,"silver",Entity)

In [ ]:
%sql
select * from silver.dimpaymenttypes